In [0]:
%run ../../00_common/data_utils

In [0]:
# Master 子表配置：(表名, scon_id 列名, mrkt_code 列名)
MASTER_SUB_TABLES = [
    ("t_master_address", "scad_scon_id", "scad_mrkt_code"),
    ("t_master_phone", "scph_scon_id", "scph_mrkt_code"),
    ("t_master_emedia", "scme_scon_id", "scme_mrkt_code"),
    ("t_master_optin", "scop_scon_id", "scop_mrkt_code"),
    ("t_master_auxiliary_attribute", "scaa_scon_id", "scaa_mrkt_code"),
    ("t_master_crossbrand_optin", "scbo_scon_id", "scbo_mrkt_code"),
    ("t_master_custom_attributes", "sccu_scon_id", "sccu_mrkt_code"),
    ("t_master_consumer_group", "scgr_scon_id", "scgr_mrkt_code"),
    ("t_master_hair_concerns", "schc_scon_id", "schc_mrkt_code"),
    ("t_master_hair_type", "scht_scon_id", "scht_mrkt_code"),
    ("t_master_hobby", "scho_scon_id", "scho_mrkt_code"),
    ("t_master_makeup_concerns", "scmc_scon_id", "scmc_mrkt_code"),
    ("t_master_notes", "scno_scon_id", "scno_mrkt_code"),
    ("t_master_program", "scpr_scon_id", "scpr_mrkt_code"),
    ("t_master_remark", "scre_scon_id", "scre_mrkt_code"),
    ("t_master_skin_concerns", "scsc_scon_id", "scsc_mrkt_code"),
    ("t_master_terms", "scte_scon_id", "scte_mrkt_code"),
]

# Clean 子表配置：(表名, srcc_id 列名, mrkt_code 列名)
CLEAN_SUB_TABLES = [
    ("t_clean_address", "srca_srcc_id", "srca_mrkt_code"),
    ("t_clean_auxiliary_attribute", "sraa_srcc_id", "sraa_mrkt_code"),
    ("t_clean_consumergroup", "srcg_srcc_id", "srcg_mrkt_code"),
    ("t_clean_crossbrand_optin", "srbo_srcc_id", "srbo_mrkt_code"),
    ("t_clean_custom_attributes", "srat_srcc_id", "srat_mrkt_code"),
    ("t_clean_emedia", "srce_srcc_id", "srce_mrkt_code"),
    ("t_clean_hair_concerns", "srhc_srcc_id", "srhc_mrkt_code"),
    ("t_clean_hair_type", "srht_srcc_id", "srht_mrkt_code"),
    ("t_clean_hobby", "srhb_srcc_id", "srhb_mrkt_code"),
    ("t_clean_makeup_concerns", "srmc_srcc_id", "srmc_mrkt_code"),
    ("t_clean_notes", "srno_srcc_id", "srno_mrkt_code"),
    ("t_clean_optin", "srco_srcc_id", "srco_mrkt_code"),
    ("t_clean_phone", "srcp_srcc_id", "srcp_mrkt_code"),
    ("t_clean_program", "srpg_srcc_id", "srpg_mrkt_code"),
    ("t_clean_remark", "srcr_srcc_id", "srcr_mrkt_code"),
    ("t_clean_skin_concerns", "srsk_srcc_id", "srsk_mrkt_code"),
    ("t_clean_terms", "srct_srcc_id", "srct_mrkt_code"),
]

In [0]:
def load_anonymization_keys(task_id):
    """
    读取 t_mdm_anonymization_log 当前 task_id，
    关联 t_master_consumer 获取 scon_id / business key / consumermdmkey。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    golden_db = get_env_config('golden_consumer_master_database')

    log_table = f"{anonymization_db}.t_mdm_anonymization_log"
    master_table = f"{golden_db}.t_master_consumer"

    log_df = (
        spark.table(log_table)
        .where(
            (F.col("status") == ANON_STATUS_IN_PROGRESS) &
            (F.col("task_id") == task_id)
        )
        .select(
            F.col("MarketCode"),
            F.col("BrandCode"),
            F.col("SourceSystemCode"),
            F.col("ConsumerId"),
            F.col("New_UniversalKey")
        )
        .distinct()
    )

    log_df = log_df.checkpoint(eager=True)

    log_count = log_df.count()
    print(f"load_anonymization_keys: found {log_count} log rows")
    if log_count == 0:
        print("load_anonymization_keys: no anonymization log rows, return empty keys")
        return spark.createDataFrame(
            [],
            "scon_id LONG, scon_mrkt_code STRING, scon_brnd_code STRING, "
            "scon_srcs_code STRING, scon_consumerid STRING, consumermdmkey STRING"
        )

    # 先用 business key 对 master 做 semi join，减少扫描量
    log_business_keys = (
        log_df
        .select(
            F.col("MarketCode").alias("scon_mrkt_code"),
            F.col("BrandCode").alias("scon_brnd_code"),
            F.col("SourceSystemCode").alias("scon_srcs_code"),
            F.col("ConsumerId").alias("scon_consumerid")
        )
        .distinct()
    )

    master_df = (
        spark.table(master_table)
        .join(
            F.broadcast(log_business_keys),
            ["scon_mrkt_code", "scon_brnd_code", "scon_srcs_code", "scon_consumerid"],
            "semi"
        )
    )

    keys_df = (
        log_df.alias("log")
        .join(
            master_df.alias("m"),
            (F.col("log.MarketCode") == F.col("m.scon_mrkt_code")) &
            (F.col("log.BrandCode") == F.col("m.scon_brnd_code")) &
            (F.col("log.SourceSystemCode") == F.col("m.scon_srcs_code")) &
            (F.col("log.ConsumerId") == F.col("m.scon_consumerid")) &
            (F.col("log.New_UniversalKey") == F.col("m.consumermdmkey")),
            "inner"
        )
        .select(
            F.col("m.scon_id"),
            F.col("m.scon_mrkt_code"),
            F.col("m.scon_brnd_code"),
            F.col("m.scon_srcs_code"),
            F.col("m.scon_consumerid"),
            F.col("m.consumermdmkey")
        )
        .distinct()
    )

    count = keys_df.count()
    print(f"load_anonymization_keys: resolved {count} master rows")
    return keys_df

In [0]:
def delete_master_tables(source_df):
    """
    删除 master 层业务表（17 张子表 + t_master_consumer 主表）。
    匹配条件：{prefix}_SCON_ID = source.scon_id AND {prefix}_MRKT_CODE = source.scon_mrkt_code。
    source_df 需已包含去重后的 scon_id、scon_mrkt_code。
    顺序：先删子表，再删主表，以满足外键约束。
    """
    golden_db = get_env_config('golden_consumer_master_database')

    if source_df.isEmpty():
        print("delete_master_tables: no keys, skip")
        return

    # 删master子表
    for table_name, fk_col, mrkt_col in MASTER_SUB_TABLES:
        full_table_name = f"{golden_db}.{table_name}"
        try:
            (
                DeltaTable.forName(spark, full_table_name).alias("target")
                .merge(
                    source_df.alias("source"),
                    f"target.{fk_col} = source.scon_id AND target.{mrkt_col} = source.scon_mrkt_code"
                )
                .whenMatchedDelete()
                .execute()
            )
            print(f"delete_master_tables: processed {full_table_name}")
        except Exception as e:
            raise RuntimeError(f"delete_master_tables failed on {full_table_name}: {e}") from e

    # 最后删 master 主表
    master_consumer_table = f"{golden_db}.t_master_consumer"
    try:
        (
            DeltaTable.forName(spark, master_consumer_table).alias("target")
            .merge(
                source_df.alias("source"),
                "target.scon_id = source.scon_id AND target.scon_mrkt_code = source.scon_mrkt_code"
            )
            .whenMatchedDelete()
            .execute()
        )
        print(f"delete_master_tables: processed {master_consumer_table}")
    except Exception as e:
        raise RuntimeError(f"delete_master_tables failed on {master_consumer_table}: {e}") from e

In [0]:
def delete_derived_tables(keys_df):
    """
    删除 derived 表 l1/l2/l3。
    按 scon_mrkt_code + consumermdmkey 删。
    """
    golden_db = get_env_config('golden_consumer_master_database')

    derived_ukey_set = (
        keys_df
        .select(
            F.col("scon_mrkt_code"),
            F.col("consumermdmkey")
        )
        .distinct()
        .filter(F.col("consumermdmkey").isNotNull())
    )

    if derived_ukey_set.isEmpty():
        print("delete_derived_tables: no ukeys, skip")
        return

    derived_tables = [
        f"{golden_db}.t_derived_consumer_l1",
        f"{golden_db}.t_derived_consumer_l2",
        f"{golden_db}.t_derived_consumer_l3",
    ]

    for table_name in derived_tables:
        try:
            (
                DeltaTable.forName(spark, table_name).alias("target")
                .merge(
                    derived_ukey_set.alias("source"),
                    """
                    target.scon_mrkt_code = source.scon_mrkt_code AND
                    target.consumermdmkey = source.consumermdmkey
                    """
                )
                .whenMatchedDelete()
                .execute()
            )
            print(f"delete_derived_tables: {table_name} processed")
        except Exception as e:
            raise RuntimeError(f"delete_derived_tables failed on {table_name}: {e}") from e

In [0]:
def delete_master_layer(keys_df):
    """
    封装 master 层删除顺序：先删 derived 表，再删 master 业务表（子表 + 主表）。
    derived 引用 master，必须先删子表再删父表以满足外键约束。
    """
    master_source_df = keys_df.select(F.col("scon_id"), F.col("scon_mrkt_code")).distinct()

    print("S3-1: delete derived tables")
    delete_derived_tables(keys_df)

    print("S3-2: delete master tables")
    delete_master_tables(master_source_df)

In [0]:
def delete_clean_tables(clean_keys_df):
    """
    删除 clean 层业务表（17 张子表 + t_clean_consumer 主表）。
    子表按 {prefix}_SRCC_ID + {prefix}_MRKT_CODE 删；
    主表按 business key（MRKT/BRND/SRCS/CONSUMERID）删。
    clean_keys_df 需已包含 match_clean_keys_df 解析后的输出列。
    顺序：先删子表，再删主表，以满足外键约束。
    """
    silver_db = get_env_config('silver_consumer_cleansed_database')

    if clean_keys_df.isEmpty():
        print("delete_clean_tables: no clean consumer matches, skip")
        return

    clean_id_df = clean_keys_df.select(F.col("srcc_id"), F.col("srcc_mrkt_code")).distinct()

    # 删 clean 子表
    for table_name, fk_col, mrkt_col in CLEAN_SUB_TABLES:
        full_table_name = f"{silver_db}.{table_name}"
        try:
            (
                DeltaTable.forName(spark, full_table_name).alias("target")
                .merge(
                    clean_id_df.alias("source"),
                    f"target.{fk_col} = source.srcc_id AND target.{mrkt_col} = source.srcc_mrkt_code"
                )
                .whenMatchedDelete()
                .execute()
            )
            print(f"delete_clean_tables: processed {full_table_name}")
        except Exception as e:
            raise RuntimeError(f"delete_clean_tables failed on {full_table_name}: {e}") from e

    # 最后删 clean 主表
    clean_consumer_table = f"{silver_db}.t_clean_consumer"
    try:
        (
            DeltaTable.forName(spark, clean_consumer_table).alias("target")
            .merge(
                clean_keys_df.alias("source"),
                """
                target.SRCC_MRKT_CODE = source.scon_mrkt_code AND
                target.SRCC_BRND_CODE = source.scon_brnd_code AND
                target.SRCC_SRCS_CODE = source.scon_srcs_code AND
                target.SRCC_CONSUMERID = source.scon_consumerid
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        print(f"delete_clean_tables: processed {clean_consumer_table}")
    except Exception as e:
        raise RuntimeError(f"delete_clean_tables failed on {clean_consumer_table}: {e}") from e

In [0]:
def delete_clean_landing(clean_id_df):
    """
    删除 landing 层 consumerlist_raw 中对应 srcc_id 的记录。
    clean_id_df 需已包含去重后的 srcc_id。
    """
    if clean_id_df.isEmpty():
        print("delete_clean_landing: no clean consumer matches, skip")
        return

    landing_path = get_env_config('bronze_path_consumer')
    try:
        (
            DeltaTable.forPath(spark, landing_path).alias("target")
            .merge(
                clean_id_df.alias("source"),
                "target.slndc_id = source.srcc_id"
            )
            .whenMatchedDelete()
            .execute()
        )
        print(f"delete_clean_landing: landing {landing_path} processed")
    except Exception as e:
        raise RuntimeError(f"delete_clean_landing failed on landing {landing_path}: {e}") from e

In [0]:
def match_clean_keys_df(keys_df):
    """
    解析 clean consumer ID，返回匹配 t_clean_consumer 的 DataFrame。
    由外层负责 checkpoint 和删除操作。
    """
    silver_db = get_env_config('silver_consumer_cleansed_database')
    clean_consumer_table = f"{silver_db}.t_clean_consumer"

    # 先用 business key 对 clean_consumer 做 semi join，减少扫描量
    clean_business_keys = (
        keys_df
        .select(
            F.col("scon_mrkt_code").alias("SRCC_MRKT_CODE"),
            F.col("scon_brnd_code").alias("SRCC_BRND_CODE"),
            F.col("scon_srcs_code").alias("SRCC_SRCS_CODE"),
            F.col("scon_consumerid").alias("SRCC_CONSUMERID")
        )
        .distinct()
    )

    clean_consumer_df = (
        spark.table(clean_consumer_table)
        .join(
            F.broadcast(clean_business_keys),
            ["SRCC_MRKT_CODE", "SRCC_BRND_CODE", "SRCC_SRCS_CODE", "SRCC_CONSUMERID"],
            "semi"
        )
    )

    return (
        F.broadcast(keys_df.alias("k"))
        .join(
            clean_consumer_df.alias("c"),
            (F.col("k.scon_mrkt_code") == F.col("c.SRCC_MRKT_CODE")) &
            (F.col("k.scon_brnd_code") == F.col("c.SRCC_BRND_CODE")) &
            (F.col("k.scon_srcs_code") == F.col("c.SRCC_SRCS_CODE")) &
            (F.col("k.scon_consumerid") == F.col("c.SRCC_CONSUMERID")),
            "inner"
        )
        .select(
            F.col("k.scon_mrkt_code"),
            F.col("k.scon_brnd_code"),
            F.col("k.scon_srcs_code"),
            F.col("k.scon_consumerid"),
            F.col("c.SRCC_ID").alias("srcc_id"),
            F.col("c.SRCC_MRKT_CODE").alias("srcc_mrkt_code")
        )
        .distinct()
    )

In [0]:
def delete_cbr_datasets(keys_df):
    """
    删除 CBR dataset 表数据。
    按 keys 的 scon_mrkt_code + consumermdmkey 对应 CBR 表的 MarketCode + MDMKey。
    """
    combine_db = get_env_config('golden_consumer_combine_database')

    cbr_tables = [
        f"{combine_db}.t_cbr_dataset",
        f"{combine_db}.t_cbr_withoutpii_dataset",
        f"{combine_db}.t_cbrdrjart_dataset",
    ]

    source_df = (
        keys_df
        .select(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("consumermdmkey").alias("MDMKey")
        )
        .filter(F.col("MDMKey").isNotNull())
        .distinct()
    )

    if source_df.isEmpty():
        print("delete_cbr_datasets: no keys, skip")
        return

    for table_name in cbr_tables:
        try:
            (
                DeltaTable.forName(spark, table_name).alias("target")
                .merge(
                    source_df.alias("source"),
                    "target.MarketCode = source.MarketCode AND target.MDMKey = source.MDMKey"
                )
                .whenMatchedDelete()
                .execute()
            )
            print(f"delete_cbr_datasets: processed {table_name}")
        except Exception as e:
            raise RuntimeError(f"delete_cbr_datasets failed on {table_name}: {e}") from e

In [0]:
def delete_transaction_and_membership(keys_df):
    """
    删除 transaction 主表、transaction dataset、membership mapping log。
    按 keys 的 business key 删。
    - t_transaction_master: tran_mrkt_code/brnd/srcs + tran_mapping_consumer_id
    - t_membership_mapping_log: SRCC_MRKT_CODE/BRND/SRCS/CONSUMERID
    - t_transaction_master_dataset: 通过 t_transaction_master.tran_id 关联删除
    """
    master_db = get_env_config('golden_consumer_master_database')
    combine_db = get_env_config('golden_consumer_combine_database')

    tx_source = (
        keys_df
        .select(
            F.col("scon_mrkt_code").alias("tran_mrkt_code"),
            F.col("scon_brnd_code").alias("tran_brnd_code"),
            F.col("scon_srcs_code").alias("tran_srcs_code"),
            F.col("scon_consumerid").alias("tran_mapping_consumer_id")
        )
        .distinct()
    )

    membership_source = (
        keys_df
        .select(
            F.col("scon_mrkt_code").alias("SRCC_MRKT_CODE"),
            F.col("scon_brnd_code").alias("SRCC_BRND_CODE"),
            F.col("scon_srcs_code").alias("SRCC_SRCS_CODE"),
            F.col("scon_consumerid").alias("SRCC_CONSUMERID")
        )
        .distinct()
    )

    if tx_source.isEmpty():
        print("delete_transaction_and_membership: no tx keys, skip")
        return

    # 先获取这批人要删的 tran_id，用于删 t_transaction_master_dataset
    tx_id_df = (
        spark.table(f"{master_db}.t_transaction_master")
        .alias("t")
        .join(
            F.broadcast(tx_source.alias("s")),
            (F.col("t.tran_mrkt_code") == F.col("s.tran_mrkt_code")) &
            (F.col("t.tran_brnd_code") == F.col("s.tran_brnd_code")) &
            (F.col("t.tran_srcs_code") == F.col("s.tran_srcs_code")) &
            (F.col("t.tran_mapping_consumer_id") == F.col("s.tran_mapping_consumer_id")),
            "inner"
        )
        .select(F.col("t.tran_id"))
        .distinct()
    )

    tx_id_df = tx_id_df.checkpoint(eager=True)
    tx_id_count = tx_id_df.count()
    print(f"delete_transaction_and_membership: found {tx_id_count} tx ids")

    # 先删 t_transaction_master_dataset（子表），通过 tran_id
    tx_dataset_table = f"{combine_db}.t_transaction_master_dataset"
    if tx_id_count > 0:
        try:
            (
                DeltaTable.forName(spark, tx_dataset_table).alias("target")
                .merge(
                    tx_id_df.alias("source"),
                    "target.tran_id = source.tran_id"
                )
                .whenMatchedDelete()
                .execute()
            )
            print("delete_transaction_and_membership: t_transaction_master_dataset processed")
        except Exception as e:
            raise RuntimeError(f"delete_transaction_and_membership failed on {tx_dataset_table}: {e}") from e

    # 最后删 t_transaction_master（父表）
    tx_master_table = f"{master_db}.t_transaction_master"
    try:
        (
            DeltaTable.forName(spark, tx_master_table).alias("target")
            .merge(
                tx_source.alias("source"),
                """
                target.tran_mrkt_code = source.tran_mrkt_code AND
                target.tran_brnd_code = source.tran_brnd_code AND
                target.tran_srcs_code = source.tran_srcs_code AND
                target.tran_mapping_consumer_id = source.tran_mapping_consumer_id
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        print("delete_transaction_and_membership: t_transaction_master processed")
    except Exception as e:
        raise RuntimeError(f"delete_transaction_and_membership failed on {tx_master_table}: {e}") from e

    membership_table = f"{combine_db}.t_membership_mapping_log"
    try:
        (
            DeltaTable.forName(spark, membership_table).alias("target")
            .merge(
                membership_source.alias("source"),
                """
                target.SRCC_MRKT_CODE = source.SRCC_MRKT_CODE AND
                target.SRCC_BRND_CODE = source.SRCC_BRND_CODE AND
                target.SRCC_SRCS_CODE = source.SRCC_SRCS_CODE AND
                target.SRCC_CONSUMERID = source.SRCC_CONSUMERID
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        print("delete_transaction_and_membership: t_membership_mapping_log processed")
    except Exception as e:
        raise RuntimeError(f"delete_transaction_and_membership failed on {membership_table}: {e}") from e

In [0]:
def delete_records(task_id):
    """
    S3(master + derived) -> S2(clean) -> S1(landing) -> S4(CBR datasets) -> S5(transaction & membership)
    按以上顺序删除当前任务的所有相关记录。
    """
    keys_df = load_anonymization_keys(task_id)
    keys_df = keys_df.checkpoint(eager=True)

    keys_count = keys_df.count()
    if keys_count == 0:
        print("No anonymization keys for this task, nothing to delete")
        return

    print(f"[processing] {keys_count} keys")

    print("S3: delete master tables")
    delete_master_layer(keys_df)

    print("S2: resolve clean consumer matches")
    clean_keys_df = match_clean_keys_df(keys_df)
    clean_keys_df = clean_keys_df.checkpoint(eager=True)

    print("S2: delete clean tables")
    delete_clean_tables(clean_keys_df)

    clean_id_df = clean_keys_df.select(F.col("srcc_id")).distinct()
    print("S1: delete clean landing records")
    delete_clean_landing(clean_id_df)

    print("S4: delete CBR datasets")
    delete_cbr_datasets(keys_df)

    print("S5: delete transaction and membership records")
    delete_transaction_and_membership(keys_df)

    print("[completed]")

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

step_name = "delete_records"
step_num = "03"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    delete_records(task_id)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )